# Libraries

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
import seaborn as sns
sns.set_palette('pastel')
sns.set_style('whitegrid')
import nibabel as nib
import os
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from statsmodels.formula.api import ols
from sklearn.metrics import silhouette_score, normalized_mutual_info_score, rand_score, adjusted_rand_score, calinski_harabasz_score
from sklearn.cluster import KMeans
import networkx as nx
from sklearn.metrics import davies_bouldin_score
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import glm, ols
import pingouin as pg
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.multitest import fdrcorrection

# set seaborn palette to colorblind
sns.set_palette("colorblind")

# hide warnings
import warnings
warnings.filterwarnings("ignore")

# Utilities

In [ ]:
def calculate_effect_size(data, group_col='kmeans_2_consensus', value_col='value'):
    # calculate the mean and standard deviation for each group
    group_1 = data[data[group_col] == 0][value_col]
    group_2 = data[data[group_col] == 1][value_col]
    mean_1 = group_1.mean()
    mean_2 = group_2.mean()
    std_1 = group_1.std()
    std_2 = group_2.std()
    n1 = len(group_1)
    n2 = len(group_2)
    
    # calculate Cohen's d
    pooled_std = np.sqrt(((n1 - 1) * std_1**2 + (n2 - 1) * std_2**2) / (n1 + n2 - 2))
    cohen_d = (mean_1 - mean_2) / pooled_std
    
    return cohen_d

# Data Wrangling

In [ ]:
df = pd.read_csv('/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/midb61_meanFC_clusters_2026-04-28_11-07.csv')
motion_df = pd.read_csv('/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/motion_QA_results.csv')
df = pd.merge(df, motion_df, on='src_subject_id')
for col in df.columns:
    print(col)

# Calculate INR for SES Analyses

In [ ]:
income_medians = {
    1.0: 2500,
    2.0: 8500,
    3.0: 14000,
    4.0: 20500,
    5.0: 30000,
    6.0: 42500,
    7.0: 62500,
    8.0: 87500,
    9.0: 150000,
    10.0: 250000
}

df["income_median"] = df["demo_comb_income_v2"].map(income_medians)

# 2017 Federal Poverty Guidelines
poverty_lines = {
    1.0: 12060,
    2.0: 16240,
    3.0: 20420,
    4.0: 24600,
    5.0: 28780,
    6.0: 32960,
    7.0: 37140,
    8.0: 41320
}

# For households > 8, add $4,180 per additional person
def get_poverty_line(hh_size):
    if pd.isna(hh_size):
        return np.nan
    if hh_size < 1:
        return np.nan
    if hh_size <= 8:
        return poverty_lines[int(hh_size)]
    else:
        extra = (int(hh_size) - 8) * 4180
        return poverty_lines[8] + extra

# Compute poverty line for each row
df["poverty_line_2017"] = df["demo_roster_v2"].apply(get_poverty_line)

# Compute income-to-needs ratio (INR)
df["inr"] = df["income_median"] / df["poverty_line_2017"]
df["inr"] = df["inr"].replace([np.inf, -np.inf], np.nan)

print(df[["demo_comb_income_v2", "demo_roster_v2", "poverty_line_2017", "inr"]].head())


# EF Measure Residualization

In [ ]:
# residualize RAVLT and SST meaures to account for age and sex
for measure in ['ravlt_immediate_2y', 'ravlt_short_delay_2y','ravlt_long_delay_2y', 'tfmri_sst_all_beh_total_issrt', 'nihtbx_picvocab_uncorrected_2y','nihtbx_reading_uncorrected_2y', 'nihtbx_flanker_uncorrected_2y', 'nihtbx_picture_uncorrected_2y']:
    # convert 555, 777, 999 to NaN
    df[measure] = df[measure].replace([555, 777, 888, 999], np.nan)
    temp_df = df[['src_subject_id', measure, 'interview_age', 'demo_sex_v2']].dropna()
    X = temp_df[['interview_age', 'demo_sex_v2']]
    y = temp_df[measure]
    model = LinearRegression().fit(X, y)
    temp_df[f'{measure}_resid'] = y - model.predict(X)
    df = pd.merge(df, temp_df[['src_subject_id', f'{measure}_resid']], on='src_subject_id', how='left')

# Comparing groups on EF measures

In [ ]:
cog_measures = ['nihtbx_picvocab_uncorrected_2y_resid','nihtbx_reading_uncorrected_2y_resid', 'nihtbx_flanker_uncorrected_2y_resid', 'nihtbx_picture_uncorrected_2y_resid', 'ravlt_immediate_2y_resid', 'ravlt_short_delay_2y_resid','ravlt_long_delay_2y_resid', 'tfmri_sst_all_beh_total_issrt_resid']
for measure in cog_measures:
    # remove all 777, 555, and 999 values from the measure
    df[measure] = df[measure].replace([777, 555, 888, 999], np.nan)

In [ ]:
cat_measures = ['tanner_stage', 'site_id_l', 'ehi1b'] 
cont_measures = ['inr']

def clean_data(df, measure):
    """Replace invalid codes with NaN and drop missing."""
    return df[[ 'kmeans_2_consensus', measure ]].replace([777, 555, 888, 999], np.nan).dropna()

def run_ttest(df, measure, bonferroni=None):
    """Run Welch’s t-test and print results."""
    cluster_1 = df[df['kmeans_2_consensus'] == 0][measure]
    cluster_2 = df[df['kmeans_2_consensus'] == 1][measure]
    t_stat, p_val = stats.ttest_ind(cluster_1, cluster_2, equal_var=False)
    
    sig_thresh = 0.05 / bonferroni if bonferroni else 0.05
    significance = "Significant" if p_val < sig_thresh else "No significant"
    
    print(f"{significance} difference in {measure} between clusters: "
          f"t_stat = {t_stat:.3f}, p-value = {p_val:.7f}")
    
    cohen_d = calculate_effect_size(df, group_col='kmeans_2_consensus', value_col=measure)
    print(f"Cohen's d = {cohen_d:.3f}\n")

def run_chi2(df, measure):
    """Run chi-square test and print results."""
    contingency = pd.crosstab(df['kmeans_2_consensus'], df[measure])
    chi2, p_val, dof, expected = stats.chi2_contingency(contingency)
    cramer_v = np.sqrt(chi2 / (df.shape[0] * (min(contingency.shape)-1)))
    
    significance = "Significant" if p_val < 0.05 else "No significant"
    print(f"{significance} difference in {measure} between clusters: "
          f"chi2 = {chi2:.3f}, p-value = {p_val:.7f}")
    print(f"Cramér's V = {cramer_v:.3f}\n")

for measure in cog_measures:
    temp_df = clean_data(df, measure)
    run_ttest(temp_df, measure, bonferroni=len(cog_measures))

for measure in cat_measures:
    temp_df = clean_data(df, measure)
    run_chi2(temp_df, measure)

for measure in cont_measures:
    temp_df = clean_data(df, measure)
    run_ttest(temp_df, measure)


# Network FC - EF  and INR Significant Correlations

In [ ]:
from scipy import stats

# parameters
n = len(df)  # sample size
n_tests = len(cog_measures + ['inr']) * 2 * 28 # two hemispheres and 28 networks, 8 cognitive measures plus INR
alpha = 0.05
trend_alpha = 0.001
bonferroni_alpha = alpha / n_tests

# degrees of freedom
dfree = n - 2

# two-tailed critical t-value
t_crit = stats.t.ppf(1 - bonferroni_alpha / 2, dfree)
trend_t_crit = stats.t.ppf(1 - trend_alpha / 2, dfree)

# convert to r critical
r_critical = np.sqrt(t_crit**2 / (t_crit**2 + dfree))
r_trend = np.sqrt(trend_t_crit**2 / (trend_t_crit**2 + dfree))

print("Sample size:", n)
print("Bonferroni alpha:", bonferroni_alpha)
print("Critical r:", r_critical)
print("Trend alpha:", trend_alpha)
print("Trend critical r:", r_trend)

In [ ]:
results = []

resid_cols = [col for col in df.columns if 'resid' in col and 'fz' in col and 'full' not in col]

for measure in cog_measures + ['inr']:
    for col in resid_cols:
        temp_df = df[[col, measure]].dropna()
        r, p = stats.pearsonr(temp_df[col], temp_df[measure])
        results.append({
            'measure': measure,
            'col': col,
            'r': r,
            'p': p,
            'n': len(temp_df)
        })

results_df = pd.DataFrame(results)

n_tests = len(results_df)

# Bonferroni correction
results_df['p_bonf'] = results_df['p'] * n_tests
results_df['sig_bonf'] = results_df['p_bonf'] < 0.05

# FDR correction
reject, p_fdr = fdrcorrection(results_df['p'], alpha=0.05)
results_df['p_fdr'] = p_fdr
results_df['sig_fdr'] = reject

# Bonferroni results
sig_cols = []

print("\nBonferroni significant correlations:")

bonf_results = results_df[results_df['sig_bonf']].sort_values('p_bonf')

for _, row in bonf_results.iterrows():
    print(
        f"{row['col']} vs {row['measure']}: "
        f"r = {row['r']:.4f}, p = {row['p']:.4g}, p_bonf = {row['p_bonf']:.4g}, n = {row['n']}"
    )
    sig_cols.append(row['col'])

# FDR results
print("\nFDR significant correlations:")

fdr_results = results_df[results_df['sig_fdr']].sort_values('p_fdr')

for _, row in fdr_results.iterrows():
    print(
        f"{row['col']} vs {row['measure']}: "
        f"r = {row['r']:.4f}, p = {row['p']:.4g}, p_fdr = {row['p_fdr']:.4g}, n = {row['n']}"
    )

print("Correlation between left pMTG - LANG FC and picture vocabulary:", results_df[(results_df['col'] == 'VAN_left_L_fz_resid') & (results_df['measure'] == 'nihtbx_picvocab_uncorrected_2y_resid')][['r', 'p']])
print("Sample size range:", results_df['n'].min(), "-", results_df['n'].max())

In [ ]:
plot_df = results_df[results_df["sig_fdr"]].copy()

plot_df["network"] = plot_df["col"].str.split("_").str[0]

plot_df["network"] = plot_df["network"].replace({
    "CO": "AMN",
    "Aud": "AUD",
    "Sal": "SAL"
})

seed_net_raw = plot_df["col"].str.split("_").str[0]
seed_hemi = plot_df["col"].str.split("_").str[1].str[0].str.upper()
seed_lr = plot_df["col"].str.split("_").str[2]

seed_net = seed_net_raw.replace({
    "CO": "AMN",
    "Aud": "AUD",
    "Sal": "SAL"
})

seed_net = np.where(
    (seed_net_raw == "VAN") & (seed_hemi == "L"),
    "LANG",
    seed_net
)

plot_df["network"] = seed_net
plot_df["hemi"] = seed_hemi

plot_df["seed_label"] = np.where(
    seed_net_raw == "VAN",
    seed_net + " - " + seed_lr + " pMTG FC",
    seed_hemi + " " + seed_net + " - " + seed_lr + " pMTG FC"
)

measure_map = {
    "tfmri_sst_all_beh_total_issrt": "SST ISSRT",
    "ravlt_immediate_2y": "RAVLT Immediate",
    "ravlt_short_delay_2y": "RAVLT Short Delay",
    "ravlt_long_delay_2y": "RAVLT Long Delay",
    "nihtbx_picvocab_uncorrected_2y": "Picture Vocabulary",
    "nihtbx_reading_uncorrected_2y": "Oral Reading Recognition",
    "nihtbx_flanker_uncorrected_2y": "Flanker",
    "nihtbx_picture_uncorrected_2y": "Picture Sequence Memory",
    "inr": "Income-to-Needs Ratio"
}

plot_df["measure_base"] = (
    plot_df["measure"]
    .str.replace("_2y_resid", "_2y", regex=False)
    .str.replace("_resid", "", regex=False)
)

plot_df["measure_label"] = plot_df["measure_base"].map(measure_map).fillna(plot_df["measure_base"])

network_colors = {
    'DMN': "#fb2e2e", 'SMd': '#40cce9', 'LANG': '#2DB6B6FF', 'TPOLE': '#2e87ae',
    'AUD': '#ce8fff', 'AMN': '#8a2edb', 'DAN': '#2efe2e', 'FP': "#cdcd00ff",
    'MTL': '#89fd89', 'PMN': '#2e2eff', 'PON': "#a29c9cff", 'SAL': '#2e2e2e',
    'VAN': "#2DB6B6FF", 'SMl': '#ffa12e', 'VIS': '#2e2eb3'
}

heat = plot_df.pivot_table(
    index="seed_label",
    columns="measure_label",
    values="r",
    aggfunc="first"
)

row_info = (
    plot_df.drop_duplicates("seed_label")
    .set_index("seed_label")
    .loc[heat.index]
)

blue_network_order = [
    "SMd",
    "SMl",
    "AUD",
    "AMN",
    "DAN",
    "VAN",
    "LANG"
]

red_network_order = [
    "DMN",
    "FP",
    "MTL",
    "PMN",
    "PON",
    "SAL"
]

network_order = blue_network_order + red_network_order

row_info["network_order"] = row_info["network"].map({
    network: i for i, network in enumerate(network_order)
})

heat = heat.loc[
    row_info.sort_values(["network_order", "hemi"]).index
]

row_info = row_info.loc[heat.index]
row_networks = row_info["network"]

fig, ax = plt.subplots(figsize=(12, max(6, 0.3 * len(heat))))

im = ax.imshow(heat, aspect="auto", cmap="coolwarm", vmin=-0.15, vmax=0.15)

ax.grid(False)

ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=45, ha="right", fontsize=11)

ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index, fontsize=11)

for y, label in enumerate(ax.get_yticklabels()):
    net = row_networks.iloc[y]
    label.set_color(network_colors.get(net, "black"))
    label.set_fontweight("bold")

for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        val = heat.iloc[i, j]
        if pd.notna(val):
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=8, fontweight="bold")

legend_handles = []

for network in network_order:
    if network in set(row_networks):
        if network == "VAN":
            label = "LANG/VAN"
        elif network == "LANG":
            continue
        else:
            label = network

        legend_handles.append(Patch(color=network_colors[network], label=label))

ax.legend(handles=legend_handles, title="Network", bbox_to_anchor=(1.25, 1), loc="upper left")

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Pearson r", fontsize=12)

ax.set_title("Significant Brain-Behavior Correlations (FDR Corrected)", fontsize=18)
ax.set_xlabel("Cognitive Measure", fontsize=16)
ax.set_ylabel("pMTG - Network Functional Connectivity Measure", fontsize=16)

plt.tight_layout()
plt.show()

# Subtype Sizes

In [ ]:
# count number of individuals in each group
louvain_group_counts = df['kmeans_2_consensus'].value_counts()
print(louvain_group_counts)

kmeans_group_counts = df['louvain_consensus'].value_counts()
print(kmeans_group_counts)

infomap_group_counts = df['infomap_community_2'].value_counts()
print(infomap_group_counts)

# Mediation Models and SES

In [ ]:
print("Mediation Models Testing INR -> FC -> Cognitive Measures")

df = df.loc[:, ~df.columns.duplicated()].copy()

sig_pairs = bonf_results[["col", "measure"]].drop_duplicates()

summary_results = []

for _, row in sig_pairs.iterrows():

    mediator = row["col"]
    task = row["measure"]

    if len({"inr", mediator, task}) < 3:
        continue

    model_df = pd.DataFrame({
        "inr": df["inr"],
        mediator: df[mediator],
        task: df[task]
    }).dropna().copy()

    med = pg.mediation_analysis(
        data=model_df,
        x="inr",
        m=mediator,
        y=task,
        alpha=0.05,
        n_boot=5000
    )

    a_row = med.iloc[0]
    b_row = med.iloc[1]
    total_row = med.loc[med["path"].eq("Total")].iloc[0]
    direct_row = med.loc[med["path"].eq("Direct")].iloc[0]
    indirect_row = med.loc[med["path"].str.contains("Indirect", case=False, na=False)].iloc[0]

    percent_mediated = (indirect_row["coef"] / total_row["coef"]) * 100

    print(f"\nTask: {task}")
    print(f"Mediator: {mediator}")
    print(med)

    if indirect_row["sig"] == "Yes":
        print("Significant mediation")
        print(f"Percent mediated: {percent_mediated:.2f}%")

        summary_results.append({
            "x": "inr",
            "task": task,
            "mediator": mediator,
            "n": len(model_df),
            "a": a_row["coef"],
            "b": b_row["coef"],
            "direct_effect": direct_row["coef"],
            "indirect_effect": indirect_row["coef"],
            "total_effect": total_row["coef"],
            "percent_mediated": percent_mediated,
            "indirect_pval": indirect_row["pval"],
            "indirect_ci_lower": indirect_row["CI[2.5%]"],
            "indirect_ci_upper": indirect_row["CI[97.5%]"]
        })

summary_df = pd.DataFrame(summary_results)

print("\nSummary of Significant Mediation Results:")
print(summary_df.sort_values("indirect_pval"))

# Comparing Clustering Algorithms

In [ ]:
# make heat matrix of louvain_consensus, infomap, and kmeans 2 results
subtype_cols = ['louvain_consensus', 'infomap_community_2', 'kmeans_2_consensus']
# calcuate Rand Index between each column in subtype_cols
matrix = np.zeros((len(subtype_cols), len(subtype_cols)))
for i in range(len(subtype_cols)):
    for j in range(len(subtype_cols)):
        rand = rand_score(df[subtype_cols[i]], df[subtype_cols[j]])
        matrix[i, j] = rand
labels = ['Louvain', 'Infomap', 'K-Means']
sns.heatmap(matrix, annot=True, cmap= 'Blues', xticklabels=labels, yticklabels=labels, fmt=".3f", cbar=False, annot_kws={"size": 16, "font": "Arial"})

louvain_perm_cols = [col for col in df.columns if 'louvain_community' in col and 'consensus' not in col and 'subtypes' not in col]
print(len(louvain_perm_cols))
louvain_rand_values = []
for i in range(len(louvain_perm_cols)):
    for j in range(i + 1, len(louvain_perm_cols)):
        col_i = louvain_perm_cols[i]
        col_j = louvain_perm_cols[j]
        rand = rand_score(df[col_i], df[col_j])
        louvain_rand_values.append(rand)
mean_rand = np.mean(louvain_rand_values)
print(f'Mean Rand Index between louvain permutation columns: {mean_rand:.44f}')

# for kmeans_perm columns, do the same as above
kmeans_perm_cols = [col for col in df.columns if 'kmeans' in col and '_run_' in col and 'consensus' not in col and 'subtypes' not in col]
print(len(kmeans_perm_cols))
kmeans_rand_values = []
for i in range(len(kmeans_perm_cols)):
    for j in range(i + 1, len(kmeans_perm_cols)):
        col_i = kmeans_perm_cols[i]
        col_j = kmeans_perm_cols[j]
        rand = rand_score(df[col_i], df[col_j])
        kmeans_rand_values.append(rand)
mean_rand = np.mean(kmeans_rand_values)
print(f'Mean Rand Index between kmeans permutation columns: {mean_rand:.44f}')

print("Louvain Permutation Rand Scores:")
print("Range:", min(louvain_rand_values), "-", max(louvain_rand_values))
print("Mean:", np.mean(louvain_rand_values))
print("SD:", np.std(louvain_rand_values))

print("KMeans Run Rand Scores:")
print("Range:", min(kmeans_rand_values), "-", max(kmeans_rand_values))
print("Mean:", np.mean(kmeans_rand_values))
print("SD:", np.std(kmeans_rand_values))